# Comprehensive analysis and comparison of all two-module RNN runs

This notebook discovers all existing two-module RNN simulation runs, loads their data, and runs **exhaustive analysis**: transfer/interference, error, loss, PCA, principal angles, and (where available) module dynamics (dynspec-style). Results are compared across runs with a summary table and figures.

Run cells in order. No training is performed here—only loading and analysis of already-saved runs.

## Methodology

- **Transfer**: Per run, per schedule (same/near/far), we compute `error_diff = initial_B_accuracy - final_A1_accuracy` (mean over first 6 B test trials minus mean over last 6 A1 test trials). We aggregate mean/SEM per (run_id, schedule) and compare across runs. Implemented via `ann.compute_transfer_anns()` and `stats.compare_transfer()`.

- **Loss**: Per run we use `ann.analyze_training_loss(ann_data)` to get mean (and std) loss over participants per schedule across training. We compare loss curves (epochs) and final loss by run.

- **Error / accuracy**: We use the same accuracy arrays and definitions as transfer (final A1, initial B); optional A2 retest summary can be added.

- **PCA**: Per run we call `ann.compute_pca_components(ann_data, variance_threshold=0.99)` to get the number of components needed for post-A and post-B hidden representations to explain 99% variance. We compare mean n_pca by run and schedule.

- **Principal angles**: Per run we call `ann.get_principal_angles(ann_data)` to compute the principal angle between A and B subspaces (first 6 vs last 6 stimuli in post-B hiddens). We compare across runs.

- **Module dynamics (when available)**: If npz files contain `hiddens_per_module` (shape n_phase × n_trials × n_modules × dim), we compute per-module L2 norm averaged over trials, then mean magnitude per phase per module (dynspec-style). If `hiddens_post_phase_*_trajectory` exists (nb_steps > 1), we can project trajectories into 2D PCA. If `hiddens_post_phase_*_per_module` exists, we use it for per-module magnitude or 2D PCA.

- **Run discovery**: We load `experiments.json`, filter conditions with `arch == "two_module_rnn"`, and compute `run_id = build_run_id(condition)` for each. We scan `data/simulations/` and keep folders that either match one of these run_ids or have names starting with `two_module_rnn`. For each folder we optionally read `settings.json` for condition labels.

## 1. Setup and paths

In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root (parent of a1b2)
project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.utils.run_config import build_run_id
from a1b2.analysis import transfer_interference as ann
from a1b2.analysis import stats
from a1b2.utils import figure_utils
from a1b2.utils.figure_settings import schedule_colours, condition_order, task_colours

data_folder = project_root / "data"
sim_folder = data_folder / "simulations"
config_path = project_root / "a1b2" / "models" / "experiments.json"
with open(config_path, "r") as f:
    settings = json.load(f)
print("Project root:", project_root)
print("Simulations folder:", sim_folder)

## 2. Discover two-module RNN runs

Load experiments.json, compute run_id for each two_module condition, and list simulation folders that exist. Build list of (run_id, path, condition_dict) for runs we have.

In [ ]:
with open(config_path, "r") as f:
    settings = json.load(f)

two_module_conditions = [c for c in settings["conditions"] if c.get("arch") == "two_module_rnn"]
expected_run_ids = {build_run_id(c) for c in two_module_conditions}

# All dirs in simulations that are two_module runs (by name or by expected run_id)
existing_dirs = []
if sim_folder.exists():
    for d in sim_folder.iterdir():
        if d.is_dir() and (d.name.startswith("two_module_rnn") or d.name in expected_run_ids):
            existing_dirs.append(d.name)

# For each existing dir, try to get condition from settings.json
runs = []  # list of (run_id, path, condition_dict or None)
for run_id in sorted(existing_dirs):
    path = sim_folder / run_id
    condition_dict = None
    settings_path = path / "settings.json"
    if settings_path.exists():
        with open(settings_path, "r") as f:
            run_settings = json.load(f)
            condition_dict = run_settings.get("condition")
    runs.append((run_id, path, condition_dict))

print(f"Found {len(runs)} two_module run(s):")
for run_id, path, cond in runs:
    n_npz = len(list(path.glob("*.npz"))) if path.exists() else 0
    cond_name = cond.get("name", run_id) if cond else run_id
    print(f"  {run_id}  ({n_npz} npz files)  condition: {cond_name}")

## 3. Load data (per run, with optional RNN keys)

Load ann_data for each run using `load_ann_data(..., load_rnn_extra=True)` so we get `hiddens_per_module` and trajectory keys when present. Skip runs with no or incomplete npz (missing same/near/far).

In [ ]:
all_ann_data = {}  # run_id -> ann_data dict (same/near/far)
skipped = []

for run_id, path, _ in runs:
    if not path.exists():
        skipped.append((run_id, "path does not exist"))
        continue
    ann_data = ann.load_ann_data(str(path), load_rnn_extra=True)
    n_same, n_near, n_far = len(ann_data["same"]), len(ann_data["near"]), len(ann_data["far"])
    if n_same == 0 and n_near == 0 and n_far == 0:
        skipped.append((run_id, "no npz files with same/near/far in filename"))
        continue
    if n_same == 0 or n_near == 0 or n_far == 0:
        skipped.append((run_id, f"incomplete: same={n_same}, near={n_near}, far={n_far}"))
    all_ann_data[run_id] = ann_data

print(f"Loaded {len(all_ann_data)} run(s).")
if skipped:
    print("Skipped:", skipped)
for run_id, data in all_ann_data.items():
    print(f"  {run_id}: same={len(data['same'])}, near={len(data['near'])}, far={len(data['far'])}")

## 4. Compute metrics (transfer, loss, PCA, principal angles)

Per-run transfer DataFrame (with run_id), loss results dict, PCA DataFrame, and principal angles DataFrame.

In [ ]:
# Transfer: add run_id and concatenate
transfer_dfs = []
for run_id, data in all_ann_data.items():
    df = ann.compute_transfer_anns(data)
    df["run_id"] = run_id
    transfer_dfs.append(df)
transfer_all = pd.concat(transfer_dfs, ignore_index=True)

# Loss per run
loss_results = {}
for run_id, data in all_ann_data.items():
    loss_results[run_id] = ann.analyze_training_loss(data)

# PCA: add run_id and concatenate
pca_dfs = []
for run_id, data in all_ann_data.items():
    df = ann.compute_pca_components(data, variance_threshold=0.99)
    df["run_id"] = run_id
    pca_dfs.append(df)
pca_all = pd.concat(pca_dfs, ignore_index=True)

# Principal angles: add run_id and concatenate
angle_dfs = []
for run_id, data in all_ann_data.items():
    df = ann.get_principal_angles(data)
    df["run_id"] = run_id
    angle_dfs.append(df)
angles_all = pd.concat(angle_dfs, ignore_index=True)

print("Transfer (first rows):")
print(transfer_all.head())
print("\nPCA (first rows):")
print(pca_all.head())
print("\nPrincipal angles (first rows):")
print(angles_all.head())

## 4b. Module dynamics (where available)

For runs that have `hiddens_per_module`, compute per-phase per-module mean L2 magnitude and store for plotting. Detect which runs have trajectory or per_module data.

In [ ]:
def activity_magnitude_per_module(participant_data):
    """Per-phase per-module mean L2 norm from hiddens_per_module. Returns (n_phase, n_modules) or None."""
    if "hiddens_per_module" not in participant_data:
        return None
    H = participant_data["hiddens_per_module"]  # (n_phase, n_trials, n_mod, dim)
    n_phase, n_trials, n_mod, dim = H.shape
    norms = np.linalg.norm(H, axis=3)  # (n_phase, n_trials, n_mod)
    return np.nanmean(norms, axis=1)  # (n_phase, n_mod)

# Per run: collect magnitude data and whether RNN extras exist
module_magnitude_by_run = {}  # run_id -> list of (schedule, mean_mag array (n_phase, n_mod))
runs_with_module_dynamics = []
runs_with_trajectory = []

for run_id, data in all_ann_data.items():
    has_per_module = False
    has_traj = False
    mags = []
    for sched in ["same", "near", "far"]:
        for i, participant_data in enumerate(data[sched]):
            mag = activity_magnitude_per_module(participant_data)
            if mag is not None:
                has_per_module = True
                mags.append((sched, mag))
            if "hiddens_post_phase_0_trajectory" in participant_data or "hiddens_post_phase_1_trajectory" in participant_data:
                has_traj = True
    if has_per_module:
        runs_with_module_dynamics.append(run_id)
        # Aggregate: mean magnitude per (phase, module) per schedule
        by_sched = {}
        for sched in ["same", "near", "far"]:
            sched_mags = [m for sc, m in mags if sc == sched]
            if sched_mags:
                by_sched[sched] = np.nanmean(np.stack(sched_mags), axis=0)  # (n_phase, n_mod)
        module_magnitude_by_run[run_id] = by_sched
    if has_traj:
        runs_with_trajectory.append(run_id)

print("Runs with module dynamics (hiddens_per_module):", runs_with_module_dynamics)
print("Runs with trajectory:", runs_with_trajectory)

## 5. Summary table

One row per run_id: condition name, sparsity, input_routing, common_readout, n_participants per schedule, mean error_diff per schedule, mean n_pca, mean principal angle, final loss per schedule, module dynamics available.

In [ ]:
# Build summary from runs list and computed metrics
summary_rows = []
for run_id, path, cond in runs:
    if run_id not in all_ann_data:
        continue
    data = all_ann_data[run_id]
    cond_name = cond.get("name", run_id) if cond else run_id
    sparsity = cond.get("sparsity", None) if cond else None
    input_routing = cond.get("input_routing", "shared") if cond else "shared"
    common_readout = cond.get("common_readout", True) if cond else True

    n_same = len(data["same"])
    n_near = len(data["near"])
    n_far = len(data["far"])

    tr = transfer_all[transfer_all["run_id"] == run_id]
    err_same = tr[tr["condition"] == "same"]["error_diff"].mean()
    err_near = tr[tr["condition"] == "near"]["error_diff"].mean()
    err_far = tr[tr["condition"] == "far"]["error_diff"].mean()

    pc = pca_all[pca_all["run_id"] == run_id]
    n_pca_postA = pc[pc["task"] == "post A"]["n_pca"].mean()
    n_pca_postB = pc[pc["task"] == "post B"]["n_pca"].mean()

    ang = angles_all[angles_all["run_id"] == run_id]
    mean_angle = ang["principal_angle_between"].mean()

    loss = loss_results[run_id]
    # final loss: last 10% of points
    final_loss_same = np.nanmean(loss["same"]["mean"][-max(1, len(loss["same"]["mean"]) // 10):])
    final_loss_near = np.nanmean(loss["near"]["mean"][-max(1, len(loss["near"]["mean"]) // 10):])
    final_loss_far = np.nanmean(loss["far"]["mean"][-max(1, len(loss["far"]["mean"]) // 10):])

    has_dynamics = run_id in runs_with_module_dynamics

    summary_rows.append({
        "run_id": run_id,
        "condition": cond_name,
        "sparsity": sparsity,
        "input_routing": input_routing,
        "common_readout": common_readout,
        "n_same": n_same, "n_near": n_near, "n_far": n_far,
        "err_same": err_same, "err_near": err_near, "err_far": err_far,
        "n_pca_postA": n_pca_postA, "n_pca_postB": n_pca_postB,
        "mean_principal_angle": mean_angle,
        "final_loss_same": final_loss_same, "final_loss_near": final_loss_near, "final_loss_far": final_loss_far,
        "module_dynamics": has_dynamics,
    })

summary_df = pd.DataFrame(summary_rows)
try:
    from IPython.display import display
    display(summary_df)
except ImportError:
    print(summary_df.to_string())

## 6. Comparison figures

Transfer (error_diff) by run; loss curves; n_pca by run; principal angles by run; module magnitude where available.

In [ ]:
# Short labels for x-axis (avoid long run_id strings)
def short_label(run_id, cond_dict):
    if cond_dict and "name" in cond_dict:
        name = cond_dict["name"]
        name = name.replace("two_module_rnn_50_", "").replace("two_module_rnn_50", "base")
        return name
    return run_id[:25] + "..." if len(run_id) > 25 else run_id

run_id_to_cond = {r[0]: r[2] for r in runs}
run_labels = {run_id: short_label(run_id, run_id_to_cond.get(run_id)) for run_id in all_ann_data}

In [ ]:
# Figure 1: Transfer (error_diff) by run — grouped bar
if len(all_ann_data) > 0:
    agg = transfer_all.groupby(["run_id", "condition"], as_index=False)["error_diff"].agg(["mean", "sem"])
    agg = agg.reset_index()
    run_order = list(all_ann_data.keys())
    fig, ax = plt.subplots(figsize=(max(6, len(run_order) * 0.8), 4))
    x = np.arange(len(run_order))
    w = 0.25
    for i, sched in enumerate(condition_order):
        vals = [agg[(agg["run_id"] == rid) & (agg["condition"] == sched)]["mean"].values for rid in run_order]
        vals = [v[0] if len(v) else np.nan for v in vals]
        errs = [agg[(agg["run_id"] == rid) & (agg["condition"] == sched)]["sem"].values for rid in run_order]
        errs = [e[0] if len(e) else 0 for e in errs]
        ax.bar(x + (i - 1) * w, vals, w, label=sched, color=schedule_colours[i], yerr=errs, capsize=2)
    ax.set_xticks(x)
    ax.set_xticklabels([run_labels.get(rid, rid) for rid in run_order], rotation=30, ha="right")
    ax.set_ylabel("Error diff (transfer)")
    ax.legend()
    ax.axhline(0, color="k", linewidth=0.5)
    fig.tight_layout()
    plt.show()

In [ ]:
# Figure 2: Loss curves — one subplot per run (or overlaid)
n_epochs = settings.get("n_epochs", 100)
n_runs = len(all_ann_data)
if n_runs > 0:
    n_cols = min(n_runs, 6)
    fig, axs = plt.subplots(1, n_cols, figsize=(4 * n_cols, 3), squeeze=False)
    axs = axs.flatten()
    for idx, run_id in enumerate(list(all_ann_data.keys())[:6]):
        ax = axs[idx]
        loss = loss_results[run_id]
        for s_idx, sched in enumerate(["same", "near", "far"]):
            mean_loss = loss[sched]["mean"]
            ax.plot(mean_loss, color=schedule_colours[s_idx], label=sched, alpha=0.8)
        ax.set_title(run_labels.get(run_id, run_id)[:20])
        ax.set_xlabel("Training step")
        if idx == 0:
            ax.set_ylabel("Loss (MSE)")
            ax.legend()
        ax.axvline(n_epochs * 6 * 10, color="k", linestyle="--", alpha=0.3)
        ax.axvline(2 * n_epochs * 6 * 10, color="k", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Figure 3: PCA components (n_pca) by run and task
if len(pca_all) > 0:
    fig, ax = plt.subplots(figsize=(max(5, len(all_ann_data) * 0.6), 4))
    pca_agg = pca_all.groupby(["run_id", "task"], as_index=False)["n_pca"].mean()
    run_order = list(all_ann_data.keys())
    x = np.arange(len(run_order))
    w = 0.25
    for i, (task_name, colour) in enumerate([("post A", task_colours[0]), ("post B", task_colours[1])]):
        vals = []
        for rid in run_order:
            v = pca_agg[(pca_agg["run_id"] == rid) & (pca_agg["task"] == task_name)]["n_pca"]
            vals.append(v.mean() if len(v) else np.nan)
        vals = [v if not np.isnan(v) else 0 for v in vals]
        ax.bar(x + (i - 0.5) * w, vals, w, label=task_name, color=colour)
    ax.set_xticks(x)
    ax.set_xticklabels([run_labels.get(rid, rid) for rid in run_order], rotation=30, ha="right")
    ax.set_ylabel("# PCA components (99% var)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Figure 4: Principal angles by run
if len(angles_all) > 0:
    fig, ax = plt.subplots(figsize=(max(5, len(all_ann_data) * 0.6), 4))
    ang_agg = angles_all.groupby("run_id")["principal_angle_between"].agg(["mean", "sem"]).reset_index()
    run_order = list(all_ann_data.keys())
    means = [ang_agg[ang_agg["run_id"] == rid]["mean"].values for rid in run_order]
    means = [m[0] if len(m) else np.nan for m in means]
    sems = [ang_agg[ang_agg["run_id"] == rid]["sem"].values for rid in run_order]
    sems = [s[0] if len(s) else 0 for s in sems]
    ax.bar(range(len(run_order)), means, yerr=sems, capsize=3, color="steelblue", alpha=0.8)
    ax.set_xticks(range(len(run_order)))
    ax.set_xticklabels([run_labels.get(rid, rid) for rid in run_order], rotation=30, ha="right")
    ax.set_ylabel("Principal angle (deg)")
    ax.set_xlabel("Run")
    plt.tight_layout()
    plt.show()

In [ ]:
# Figure 5: Module magnitude (dynspec-style) for runs that have hiddens_per_module
phase_labels = ["post A1", "post B", "post A2"]
if module_magnitude_by_run:
    n_plot = len(module_magnitude_by_run)
    fig, axs = plt.subplots(1, n_plot, figsize=(4 * n_plot, 3), squeeze=False)
    axs = axs[0]
    for idx, run_id in enumerate(module_magnitude_by_run):
        ax = axs[idx]
        by_sched = module_magnitude_by_run[run_id]
        for sched in ["same", "near", "far"]:
            if sched not in by_sched:
                continue
            mean_mag = by_sched[sched]  # (n_phase, n_mod)
            n_phase, n_mod = mean_mag.shape
            x = np.arange(n_phase)
            for m in range(n_mod):
                ax.plot(x, mean_mag[:, m], label=f"M{m} ({sched})", alpha=0.8)
        ax.set_xticks(np.arange(n_phase))
        ax.set_xticklabels(phase_labels[:n_phase])
        ax.set_ylabel("Mean L2 magnitude")
        ax.set_title(run_labels.get(run_id, run_id)[:22])
        if idx == 0:
            ax.legend(fontsize=6)
    plt.tight_layout()
    plt.show()
else:
    print("No runs with hiddens_per_module; skipping module magnitude plot.")

## 7. Optional: Statistical comparison across runs

Per-schedule transfer (error_diff) and principal angles: compare across run_ids (e.g. Kruskal–Wallis or ANOVA if multiple runs).

In [ ]:
# Per-run transfer stats (same as single-run notebook)
if len(all_ann_data) >= 1:
    for run_id in all_ann_data:
        tr = transfer_all[transfer_all["run_id"] == run_id]
        if len(tr) >= 3:
            run_stats = stats.compare_transfer(tr, metric_col="error_diff")
            print(f"--- {run_labels.get(run_id, run_id)} ---")
            print("Transfer (error_diff):", run_stats["descriptives"], "ANOVA p:", run_stats["anova"]["p_value"])
        print()